# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('titanic_data_updated.csv')
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [ ]:
# YOUR CODE HERE
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Deck'] = df['Cabin'].fillna('Missing').str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
484,485,yes,first,"Bishop, Mr. Dickinson H",male,25.0,1,0,11967,91.0792,B49,C,2,B
659,660,no,first,"Newell, Mr. Arthur Webster",male,58.0,0,2,35273,113.2750,D48,C,3,D
607,608,yes,first,"Daniel, Mr. Robert Williams",male,27.0,0,0,113804,30.5000,NaN,S,1,M
380,381,yes,first,"Bidois, Miss. Rosalie",female,42.0,0,0,PC 17757,227.5250,NaN,C,1,M
371,372,no,third,"Wiklund, Mr. Jakob Alfred",male,18.0,1,0,3101267,6.4958,NaN,S,2,M


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [ ]:
# YOUR CODE HERE
X = df.drop('Survived', axis = 1)
y = df['Survived']
X.shape, y.shape

((891, 13), (891,))

---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [ ]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
X_train.shape, X_test.shape

((712, 13), (179, 13))

---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [ ]:
# YOUR CODE HERE
Z_score = (X_train['Age'] - X_train['Age'].mean()) / X_train['Age'].std()
X_train = X_train[abs(Z_score) <= 3]
y_train = y_train[abs(Z_score) <= 3]

Q1 = X_train['Fare'].quantile(0.25)
Q3 = X_train['Fare'].quantile(0.75)
IQR = Q3 - Q1
minimum = max(0, Q1 - 1.5 * IQR)
maximum = Q3 + 1.5 * IQR
X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [ ]:
# YOUR CODE HERE
num_pipeline_age = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

num_pipeline_fare = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])



---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [ ]:
# YOUR CODE HERE
cat_pipeline_p3 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
])
cat_pipeline_p4 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories = [['third', 'second', 'first']])),
    ('scaler', MinMaxScaler())
])
preprocessor = ColumnTransformer([
    ('pipeline_1', num_pipeline_age, ['Age']),
    ('pipeline_2', num_pipeline_fare, ['Fare', 'Family_Size']),
    ('pipeline_3', cat_pipeline_p3, ['Embarked', 'Sex', 'Deck']),
    ('pipeline_4', cat_pipeline_p4, ['Pclass'])
], remainder='drop')


---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [ ]:
# YOUR CODE HERE

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
print(np.unique(y_train))
print(np.unique(y_test))

[0 1]
[0 1]


---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [ ]:
# YOUR CODE HERE
lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
print(y_pred[:10])

[0 0 0 0 1 1 1 0 1 0]


---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [ ]:
# YOUR CODE HERE
from sklearn.metrics import accuracy_score, precision_score, recall_score
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Accuracy  : {accuracy:.2f}")
print(f"Precision : {precision:.2f}")
print(f"Recall    : {recall:.2f}")

Accuracy  : 0.77
Precision : 0.68
Recall    : 0.75
